# Phase 4C.2 — Stage 2 Fine-Tuning Launcher

- **Mục tiêu**: Huấn luyện 15 runs Stage 2 partial fine-tuning ($N \in \{50, 100, 250\} \times 5$ seeds).
- **Runtime**: Yêu cầu NVIDIA CUDA GPU tương thích (Tesla T4, L4, V100, A100 hoặc GPU CUDA tương đương với tối thiểu 8 GB VRAM).
- **Dataset**: Tái sử dụng `phase_4c1_binary_n250_reusable.tar` từ `phase_4c1/inputs/` ở chế độ read-only.
- **Code & Operator**: Đặt tại `phase_4c2/inputs/` trên Google Drive.
- **Chế độ mặc định**: `EXECUTE = False` (chạy kiểm tra tiền kiểm `--preflight-only`, không huấn luyện).
- **Thực thi**: Đổi `EXECUTE = True` và chọn **Run all** để chạy 15 runs có khả năng resume tự động.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

EXECUTE = False

# Google Drive Paths
DRIVE_PHASE4C1_DIR = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c1")
DRIVE_DATASET_PATH = DRIVE_PHASE4C1_DIR / "inputs" / "phase_4c1_binary_n250_reusable.tar"

DRIVE_PHASE4C2_DIR = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c2")
DRIVE_INPUT_DIR = DRIVE_PHASE4C2_DIR / "inputs"
DRIVE_OUTPUT_DIR = DRIVE_PHASE4C2_DIR / "runs"

LOCAL_TRANSFER_DIR = Path("/content/forensics-stage2-transfer")
LOCAL_OUTPUT_PATH = Path("/content/phase_4c2_outputs")

CANONICAL_DATASET_SHA = "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
CANONICAL_DATASET_BYTES = 724633600

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
drive.mount("/content/drive", force_remount=False)

assert sys.platform.startswith("linux"), f"Yêu cầu Linux, nhận được: {sys.platform}"

if EXECUTE:
    assert torch.cuda.is_available(), "CUDA không khả dụng khi EXECUTE=True"
    gpu_model = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    assert vram_gb >= 8.0, f"VRAM không đủ: {vram_gb:.2f} GB (yêu cầu >= 8.0 GB)"
    x = torch.ones((32, 32), device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    assert y.is_cuda and torch.isfinite(y).all(), "CUDA smoke test failed"
    print(f"[+] GPU phát hiện: {gpu_model} ({vram_gb:.2f} GB VRAM) — Capability PASS")

# Preflight: fail-closed if inputs directory missing (DO NOT mkdir on inputs)
if not DRIVE_INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục inputs đã chuẩn bị trên Drive: {DRIVE_INPUT_DIR}")

if not DRIVE_DATASET_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset Stage 1 tại: {DRIVE_DATASET_PATH}")

# Verify dataset size and SHA-256 (streaming by chunk)
assert DRIVE_DATASET_PATH.stat().st_size == CANONICAL_DATASET_BYTES, f"Sai kích thước dataset: {DRIVE_DATASET_PATH}"
actual_ds_sha = sha256_file(DRIVE_DATASET_PATH)
assert actual_ds_sha == CANONICAL_DATASET_SHA, f"Sai SHA-256 dataset: {actual_ds_sha} != {CANONICAL_DATASET_SHA}"
print(f"[+] Dataset Stage 1 verified (read-only): {DRIVE_DATASET_PATH.name}")

# Discover Stage 2 code archive and operator script
code_archives = sorted(DRIVE_INPUT_DIR.glob("phase_4c2_code_*.tar.gz"))
if not code_archives:
    actual_entries = sorted(p.name for p in DRIVE_INPUT_DIR.iterdir())
    raise FileNotFoundError(f"Thiếu file phase_4c2_code_<sha>.tar.gz trong {DRIVE_INPUT_DIR}. Nội dung hiện có: {actual_entries}")

code_archive = code_archives[-1]
operator_script = DRIVE_INPUT_DIR / "phase_4c2_execute_all.sh"
if not operator_script.is_file():
    raise FileNotFoundError(f"Thiếu script operator: {operator_script}")

# Stage input artifacts locally via .part and os.replace
LOCAL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
for src in [code_archive, operator_script]:
    dst = LOCAL_TRANSFER_DIR / src.name
    part_dst = LOCAL_TRANSFER_DIR / f"{src.name}.part"
    if not (dst.exists() and dst.stat().st_size == src.stat().st_size):
        if part_dst.exists():
            part_dst.unlink()
        shutil.copy2(src, part_dst)
        assert part_dst.stat().st_size == src.stat().st_size, f"Lỗi copy {src.name}"
        os.replace(part_dst, dst)

(LOCAL_TRANSFER_DIR / operator_script.name).chmod(0o755)

# Setup persistent output directory only after input validation passes
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("[READY] Preflight và staging hoàn tất")

In [ ]:
op_script = LOCAL_TRANSFER_DIR / "phase_4c2_execute_all.sh"

cmd = ["bash", str(op_script)]
if EXECUTE:
    print("[RUN] Bắt đầu 15 Stage 2 partial fine-tuning runs (--execute)")
    cmd.append("--execute")
else:
    print("[PREFLIGHT] Chạy kiểm tra tiền kiểm an toàn (--preflight-only)")
    cmd.append("--preflight-only")

env = dict(os.environ)
env["DATASET_ARCHIVE"] = str(DRIVE_DATASET_PATH)
env["CODE_ARCHIVE"] = str(LOCAL_TRANSFER_DIR / code_archive.name)
env["INPUT_DIR"] = str(LOCAL_TRANSFER_DIR)

try:
    subprocess.run(cmd, env=env, check=True)
except subprocess.CalledProcessError as e:
    print(f"\n[-] FATAL: Operator thất bại với mã lỗi {e.returncode}")
    failure_reports = list(DRIVE_OUTPUT_DIR.rglob("OPERATOR_FAILURE.json"))
    console_logs = list(DRIVE_OUTPUT_DIR.rglob("operator_console.log"))
    if failure_reports:
        print("=" * 60)
        print("[*] NỘI DUNG OPERATOR_FAILURE.json:")
        print("=" * 60)
        print(failure_reports[0].read_text(encoding="utf-8"))
    if console_logs:
        log_lines = console_logs[0].read_text(encoding="utf-8", errors="replace").splitlines()
        tail_lines = log_lines[-100:]
        print("=" * 60)
        print(f"[*] 100 DÒNG CUỐI CONSOLE LOG ({len(tail_lines)}/{len(log_lines)}):")
        print("=" * 60)
        for line in tail_lines:
            print(line)
    raise RuntimeError(f"Operator execution failed with exit code {e.returncode}") from e

In [ ]:
REQUIRED_ARCHIVES = [
    "n50_stage2_results.tar.gz",
    "n100_stage2_results.tar.gz",
    "n250_stage2_results.tar.gz",
    "phase_4c2_execution_logs.tar.gz",
    "phase_4c2_all_15_runs_results.tar.gz",
]

if not EXECUTE:
    print("[INFO] Preflight-only hoàn tất thành công. Đổi EXECUTE = True và chọn Run all để huấn luyện 15 runs Stage 2.")
else:
    exec_dirs = sorted([d for d in DRIVE_OUTPUT_DIR.glob("execution_*") if d.is_dir()])
    assert exec_dirs, f"Không tìm thấy thư mục execution_* trong {DRIVE_OUTPUT_DIR}"
    final_exec_dir = exec_dirs[-1]
    download_dir = final_exec_dir / "download"
    assert download_dir.exists(), f"Thiếu download dir: {download_dir}"

    for name in REQUIRED_ARCHIVES:
        archive_path = download_dir / name
        sidecar_path = download_dir / f"{name}.sha256"
        assert archive_path.exists(), f"Thiếu archive: {archive_path}"
        assert sidecar_path.exists(), f"Thiếu sidecar: {sidecar_path}"
        assert archive_path.stat().st_size > 0, f"Archive rỗng: {archive_path}"
        expected_hash = sidecar_path.read_text(encoding="utf-8").strip().split()[0].lower()
        assert sha256_file(archive_path) == expected_hash, f"Sai SHA-256 archive: {name}"

    sample_sizes = (50, 100, 250)
    seeds = (42, 1337, 2025, 3407, 9001)
    completed_runs = 0

    for size in sample_sizes:
        for seed in seeds:
            run_dir = final_exec_dir / f"n{size}_seed_{seed}"
            receipt_path = run_dir / "run_receipt.json"
            assert receipt_path.exists(), f"Thiếu receipt: {run_dir.name}"
            with open(receipt_path, "r", encoding="utf-8") as f:
                receipt = json.load(f)
            assert receipt.get("status") == "completed", f"Chưa hoàn thành: {run_dir.name}"
            assert receipt.get("stage") == "partial_finetune", f"Sai stage: {run_dir.name}"
            assert receipt.get("sample_size") == size, f"Sai sample size: {run_dir.name}"
            assert receipt.get("seed") == seed, f"Sai seed: {run_dir.name}"
            assert receipt.get("treatment_designation") == "pre-registered partial fine-tuning protocol"
            assert receipt.get("trainable_parameters_count") == 204674
            assert receipt.get("validation_source_count") == 91
            assert receipt.get("locked_test_access") == 0
            assert receipt.get("stage1_output_writes") == 0
            completed_runs += 1

    assert completed_runs == 15, f"Kỳ vọng 15 runs, nhận được: {completed_runs}"
    print(f"[PASS] Hoàn tất và xác thực đầy đủ 15/15 runs Stage 2 trong {final_exec_dir.name}")